---
# TEIL C — Analyse & Kategorien (Meilenstein 5–6) → Ergebnis: *_analyse

# 03 – Eniac Rabattstrategie: Analyse
**Meilenstein 5–6** (siehe Projektplan)

**Projektziel:** Sind Rabatte für Eniac langfristig vorteilhaft (Position Marketing) oder schädlich für Umsatz und Positionierung (Position Vorstand/Investoren)?

**Entscheidende Frage:** Nimmt der Umsatz mit steigenden Rabatten zu?

**Inhaltsverzeichnis**
1. Import der qualitätsgeprüften Daten (Meilenstein 4)
2. Rabatt berechnen (`discount_percent`)
3. Bestellinformationen ergänzen
4. Kategorien erstellen
5. Zeitraum & Umsatz
6. Saisonale Muster
7. Meistverkaufte Produkte & Umsatz nach Kategorie
8. Rabattanalyse – Kernfrage
9. Zusatzanalysen (Preisempfehlung vs. Verkaufspreis, Bestellwert über Zeit, Produkt-Überschneidung)
10. Zusammenfassung / Entscheidungsgrundlage CEO

In [10]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.float_format', lambda x: '%.2f' % x)
pd.set_option('display.max_rows', 1000)

##1.Import der qualitätsgeprüften Daten (Meilenstein 4)

### 1.1 orderlines.csv

In [11]:
url = "https://drive.google.com/file/d/1sBf447Vvj8LbwewoyxJ8NjvQoQkyGUSw/view?usp=sharing" # orderlines.csv
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orderlines_qu = pd.read_csv(path)

### 1.2 orders.csv

In [12]:
url = "https://drive.google.com/file/d/1x4YwAPj0Q159Go4ijfRJXczppeYjkV38/view?usp=sharing" # orders.csv
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orders_qu = pd.read_csv(path)

### 1.3 products.csv

In [13]:
url = "https://drive.google.com/file/d/1S3OqtMTfGvel5r5AuWbyJW9iUw-5jmSv/view?usp=sharing" # products.csv
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
products_qu = pd.read_csv(path)

### 🔗 Brücke aus Teil B — eigene Arbeitskopie erstellen (kein Reimport von Drive)

In [14]:
orders_analyse = orders_qu.copy()
orderlines_analyse = orderlines_qu.copy()
products_analyse = products_qu.copy()

*Hinweis:* Die drei/vier Drive-Links müssen jeweils auf **unterschiedliche** Dateien zeigen (in einer früheren Version zeigten `orders_qu` und `orderlines_qu` versehentlich auf dieselbe Datei).

In [15]:
orders_analyse.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45418 entries, 0 to 45417
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   order_id      45418 non-null  int64  
 1   created_date  45418 non-null  object 
 2   total_paid    45418 non-null  float64
 3   state         45418 non-null  object 
dtypes: float64(1), int64(1), object(2)
memory usage: 1.4+ MB


In [16]:
orderlines_analyse.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60179 entries, 0 to 60178
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                60179 non-null  int64  
 1   id_order          60179 non-null  int64  
 2   product_quantity  60179 non-null  int64  
 3   sku               60179 non-null  object 
 4   unit_price        60179 non-null  float64
 5   date              60179 non-null  object 
dtypes: float64(1), int64(3), object(2)
memory usage: 2.8+ MB


In [17]:
products_analyse.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10534 entries, 0 to 10533
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   sku          10534 non-null  object 
 1   name         10534 non-null  object 
 2   desc         10534 non-null  object 
 3   price        10534 non-null  float64
 4   promo_price  10534 non-null  object 
 5   in_stock     10534 non-null  int64  
 6   type         10534 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 576.2+ KB


### 1.1 Arbeitskopien erstellen
🔗 **Wichtig:** Ab hier NUR noch mit den `_analyse`-Kopien weiterarbeiten — nicht mit `orders_qu`/`orderlines_qu`/`products_cl` direkt. So bleibt der Meilenstein-4-Datenstand unangetastet, falls ihr am Ende versehentlich mit demselben Dateinamen exportiert.

## 2. Rabatt berechnen (`discount_percent`)

In [18]:
orderlines_products_analyse = orderlines_analyse.merge(products_analyse, on='sku', how='inner')

# --- RADIKALE BEREINIGUNG FÜR BEIDE SPALTEN ---
for col in ['unit_price', 'price']:
    orderlines_products_analyse[col] = orderlines_products_analyse[col].astype(str).str.strip()
    multi_point_mask = orderlines_products_analyse[col].str.count(r'\.') > 1
    orderlines_products_analyse.loc[multi_point_mask, col] = (
        orderlines_products_analyse.loc[multi_point_mask, col].str.replace('.', '', 1)
    )

    # 3 Nachkommastellen korrigieren (z.B. 5608.689 -> 560.87)
    temp_numeric = pd.to_numeric(orderlines_products_analyse[col], errors='coerce')
    three_decimals_mask = orderlines_products_analyse[col].str.contains(r'\.\d{3}$') & temp_numeric.notna()
    if three_decimals_mask.any():
        orderlines_products_analyse.loc[three_decimals_mask, col] = round(temp_numeric / 10, 2)

# Konvertierung in echte Zahlen
orderlines_products_analyse['unit_price'] = pd.to_numeric(orderlines_products_analyse['unit_price'], errors='coerce')
orderlines_products_analyse['price'] = pd.to_numeric(orderlines_products_analyse['price'], errors='coerce')
orderlines_products_analyse = orderlines_products_analyse.dropna(subset=['unit_price', 'price'])

# --- HIER WIRD DIE FORMEL AUSGEFÜHRT ---
orderlines_products_analyse['discount_percent'] = 100 - (orderlines_products_analyse['unit_price'] * 100 / orderlines_products_analyse['price'])

# --- DIE AUSGABE: Hier machen wir das Ergebnis sichtbar ---
print("--- Die ersten Zeilen mit berechnetem Rabatt ---")
display(orderlines_products_analyse[['sku', 'unit_price', 'price', 'discount_percent']].head(20))

print("\n--- Kurze statistische Übersicht der Rabatte ---")
display(orderlines_products_analyse['discount_percent'].describe())


--- Die ersten Zeilen mit berechnetem Rabatt ---


,sku,unit_price,price,discount_percent
0,OWC0100,47.49,60.99,22.13
1,IOT0014,18.99,22.95,17.25
2,APP0700,72.19,89.00,18.89
3,PAC0929,2565.99,3209.00,20.04
4,CRU0039-A,60.90,76.99,20.90
5,PEB0015,142.49,299.99,52.50
6,BEA0065,256.49,299.95,14.49
7,SAT0010,18.99,29.99,36.68
8,SYN0139,166.24,175.99,5.54
9,LOG0191,142.49,209.00,31.82



--- Kurze statistische Übersicht der Rabatte ---


,discount_percent
count,60179.00
mean,21.91
std,20.22
min,-212.27
25%,8.27
50%,17.15
75%,28.58
max,99.98


**Formel:** Rabatt (%) = 100 − (unit_price × 100 / price)

- Positive Werte = Rabatt gegenüber Listenpreis
- Negative Werte = Verkaufspreis lag über dem aktuell gespeicherten Listenpreis (z. B. Preiserhöhung nach Kauf, oder Datenfehler)

##4.Kategorien erstellen
Ableitung aus `desc`/`name` per Keyword-Regeln, da `type` nur Codes ohne Klartext liefert.

In [19]:
alle_woerter = products_analyse['name'].fillna('').str.lower().str.split().explode()
worthaeufigkeit = alle_woerter.value_counts()
worthaeufigkeit.head(30)

,count
name,
|,4173
apple,2689
iphone,2042
case,1970
/,1960
pro,1855
-,1538
ram,1351
macbook,1289


In [20]:
import re

# ============================================================
# 1. Primäre Zuordnung über 'type' (eindeutiger als reines Keyword-Matching)
#    Basis: manuelle Stichprobenprüfung der 30 größten type-Codes
# ============================================================
type_to_category = {
    '5,74E+15': 'Computers (iMac)',
    '1282':     'Computers (MacBook)',
    '12175397': 'Storage & NAS',
    '11865403': 'Accessories & Cables',
    '2158':     'Computers (MacBook)',
    '11935397': 'Storage & NAS',
    '1,02E+12': 'Computers (MacBook)',
    '12635403': 'Accessories & Cables',
    '13835403': 'Accessories & Cables',
    '1,44E+11': 'Services',
    '1364':     'Components (RAM/SSD)',
    '1433':     'Components (RAM/SSD)',
    '12585395': 'Accessories & Cables',
    '1296':     'Monitors',
    '1325':     'Accessories & Cables',
    '5384':     'Audio',
    '12215397': 'Components (RAM/SSD)',
    '5398':     'Audio',
    '57445397': 'Components (RAM/SSD)',
    '1334':     'Network',
    '1229':     'Accessories & Cables',
    '12655397': 'Storage & NAS',
    '2449':     'Wearables',
    '12995397': 'Accessories & Cables',
    '1515':     'Accessories & Cables',
    '13615399': 'Accessories & Cables',
    '1405':     'Tablets',
    '13555403': 'Accessories & Cables',
    # '1298' und '11905404' bewusst NICHT gemappt — Stichproben zeigten gemischte
    # Inhalte ohne eindeutiges Thema. Fallen unten in den Keyword-Fallback.
}

orderlines_products_analyse['category'] = orderlines_products_analyse['type'].map(type_to_category)

# ============================================================
# 2. Fallback über Keywords MIT Wortgrenzen (behebt light/lightning, ram/frame,
#    pen/opening, ios/studios etc. — reines "in text" hatte diese Kollisionen)
# ============================================================
category_rules = [
    ('Computers (MacBook)', ['macbook']),
    ('Computers (iMac)', ['imac']),
    ('Computers (Mac Mini)', ['mac mini']),
    ('Mobile Phones', ['iphone']),
    ('Tablets', ['ipad', 'wacom', 'intuos', 'graphics tablet', 'cintiq']),
    ('Wearables', ['airpods', 'watch', 'fitbit', 'bracelet']),
    ('Audio', ['speaker', 'headphone', 'headphones', 'microphone', 'soundbar']),
    ('Monitors', ['monitor', 'display', 'cinema display']),
    ('Storage & NAS', ['nas', 'synology', 'seagate', 'raid', 'qnap', 'hard drive']),
    ('Components (RAM/SSD)', ['ram', 'ssd', 'memory card', 'sata', 'flash drive', 'pendrive']),
    ('Network', ['router', 'ethernet', 'wifi', 'wi-fi', 'bluetooth', 'airport']),
    ('Smart Home', ['philips hue', 'thermostat', 'homekit', 'motion sensor']),
    ('Drones & Action Cameras', ['gopro', 'drone', 'quadcopter']),
    ('Software', ['photoshop', 'illustrator', 'microsoft office']),
    ('Services', ['repair', 'installation', 'warranty', 'applecare']),
    ('Accessories & Cables', ['case', 'cable', 'adapter', 'charger', 'keyboard',
                              'mouse', 'lightning', 'sleeve', 'stylus']),
]

def categorize_by_keyword(text):
    for category, keywords in category_rules:
        for kw in keywords:
            if re.search(rf'\b{re.escape(kw)}\b', text):
                return category
    return 'Other'

fehlende_maske = orderlines_products_analyse['category'].isna()
name_lower_fallback = orderlines_products_analyse.loc[fehlende_maske, 'desc'].fillna('').str.lower()
orderlines_products_analyse.loc[fehlende_maske, 'category'] = name_lower_fallback.apply(categorize_by_keyword)

# ============================================================
# 3. Kontrolle
# ============================================================
print(orderlines_products_analyse['category'].value_counts())
print('Anteil Other:', round((orderlines_products_analyse['category'] == 'Other').mean() * 100, 1), '%')

category
Accessories & Cables       14438
Mobile Phones              10608
Components (RAM/SSD)        7532
Storage & NAS               7443
Computers (MacBook)         4887
Audio                       3240
Tablets                     2964
Other                       1956
Computers (iMac)            1808
Monitors                    1711
Wearables                   1595
Network                     1168
Smart Home                   345
Services                     311
Computers (Mac Mini)         166
Drones & Action Cameras        7
Name: count, dtype: int64
Anteil Other: 3.3 %


In [21]:
print(orderlines_products_analyse['category'].value_counts())
print('Anteil Other:', round((orderlines_products_analyse['category']=='Other').mean()*100, 1), '%')

category
Accessories & Cables       14438
Mobile Phones              10608
Components (RAM/SSD)        7532
Storage & NAS               7443
Computers (MacBook)         4887
Audio                       3240
Tablets                     2964
Other                       1956
Computers (iMac)            1808
Monitors                    1711
Wearables                   1595
Network                     1168
Smart Home                   345
Services                     311
Computers (Mac Mini)         166
Drones & Action Cameras        7
Name: count, dtype: int64
Anteil Other: 3.3 %


In [22]:
other_mask = orderlines_products_analyse['category'] == 'Other'
orderlines_products_analyse.loc[other_mask, :]

,id,id_order,product_quantity,sku,unit_price,date,name,desc,price,promo_price,in_stock,type,discount_percent,category
7,1119155,299564,1,SAT0010,18.99,2017-01-01 02:43:37,Satechi Aluminum Silver Mouse,Aluminum mat with ultra soft non-slip surface ...,29.99,16.992,1,1387,36.68,Other
29,1119328,297809,1,IFX0016,14.24,2017-01-01 12:01:12,iFixit Heavy Duty Suction Cups Pack 2 suction ...,Pack of 2 suction to remove the glass LCD scre...,20.99,199.408,1,14305406,32.16,Other
43,1119490,299710,1,PHI0072,66.49,2017-01-01 14:05:28,Philips Hue bulbs 9.5W White Pack 2 Starter Ki...,Kit 2 bulbs E27 + white light controller Bridge,79.95,649.903,1,11905404,16.84,Other
51,1119547,299742,2,NEA0017,21.84,2017-01-01 15:18:19,Support Netatmo MOODBOARD or Pluviometer,Netatmo support for weather station,24.99,22.99,1,11905404,12.61,Other
92,1119914,299939,1,PHI0072,66.49,2017-01-01 20:25:31,Philips Hue bulbs 9.5W White Pack 2 Starter Ki...,Kit 2 bulbs E27 + white light controller Bridge,79.95,649.903,1,11905404,16.84,Other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59968,1647418,525632,2,MAC0144,24.99,2018-03-12 17:37:58,Macally Lampcharge Lamp with 4 USB ports,Table lamp with touch control and 4 USB charge...,39.95,249.901,1,11905404,37.45,Other
60080,1648275,526426,1,TAD0001-A,124.97,2018-03-13 13:18:07,Open - Tado Smart Climate Control Intelligent AC,Reconditioned intelligent air conditioning con...,179.00,1.249.739,0,11905404,30.18,Other
60087,1648346,526455,1,MAC0144,24.99,2018-03-13 14:15:13,Macally Lampcharge Lamp with 4 USB ports,Table lamp with touch control and 4 USB charge...,39.95,249.901,1,11905404,37.45,Other
60089,1648365,526466,1,SOF0131,74.99,2018-03-13 14:37:02,Mac Parallels Desktop 13,The easiest and fastest way to run Windows on ...,79.99,749.898,0,1416,6.25,Other


*Hinweis:* Kategorie-Zuordnung ggf. verfeinern, falls `Anteil Other` zu hoch ausfällt (weitere Keywords ergänzen).

##3.Bestellinformationen ergänzen

In [23]:
orders_orderlines_products_analyse = orders_analyse.merge(
    orderlines_products_analyse, left_on="order_id", right_on="id_order", how="inner"
)
orders_orderlines_products_analyse.shape[0]

60179

In [24]:
orders_orderlines_products_analyse.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60179 entries, 0 to 60178
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          60179 non-null  int64  
 1   created_date      60179 non-null  object 
 2   total_paid        60179 non-null  float64
 3   state             60179 non-null  object 
 4   id                60179 non-null  int64  
 5   id_order          60179 non-null  int64  
 6   product_quantity  60179 non-null  int64  
 7   sku               60179 non-null  object 
 8   unit_price        60179 non-null  float64
 9   date              60179 non-null  object 
 10  name              60179 non-null  object 
 11  desc              60179 non-null  object 
 12  price             60179 non-null  float64
 13  promo_price       60179 non-null  object 
 14  in_stock          60179 non-null  int64  
 15  type              60179 non-null  object 
 16  discount_percent  60179 non-null  float6

*Hinweis:* Status-Filter (`Completed`) ist bereits in Meilenstein 4 angewendet — keine erneute Filterung nötig.

##5.Zeitraum & Umsatz

###5.1Zeitraum des Datensatzes

In [25]:
orders_analyse.loc[:, "created_date"] = pd.to_datetime(orders_analyse["created_date"])
zeitraum_start = orders_analyse["created_date"].min()
zeitraum_ende = orders_analyse["created_date"].max()
print(f"Zeitraum: {zeitraum_start} bis {zeitraum_ende}")

Zeitraum: 2017-01-01 01:51:47 bis 2018-03-14 12:03:52


###5.2Gesamtumsatz im Zeitraum

In [26]:
gesamtumsatz = orders_analyse["total_paid"].sum()
print(f"Gesamtumsatz: {gesamtumsatz:.2f}")

Gesamtumsatz: 15364514.33


###5.3Umsatz nach Jahr / Monat

In [27]:
# --- REPARATUR: Sicherstellen, dass es ein echtes Datum ist ---
orders_analyse["created_date"] = pd.to_datetime(orders_analyse["created_date"])

# Jetzt funktioniert der .dt-Zugriff fehlerfrei
orders_analyse.loc[:, "year"] = orders_analyse["created_date"].dt.year
umsatz_jahr = orders_analyse.groupby("year", as_index=False)["total_paid"].sum()
umsatz_jahr

,year,total_paid
0,2017,11920216.48
1,2018,3444297.85


In [28]:
orders_analyse.loc[:, "month"] = orders_analyse["created_date"].dt.to_period("M")
umsatz_monat = orders_analyse.groupby("month", as_index=False)["total_paid"].sum()
umsatz_monat

,month,total_paid
0,2017-01,1157930.11
1,2017-02,616113.78
2,2017-03,114599.32
3,2017-04,420213.41
4,2017-05,596554.24
5,2017-06,621297.29
6,2017-07,1010130.62
7,2017-08,710546.18
8,2017-09,847269.93
9,2017-10,1066769.02


In [29]:
orders_analyse.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45418 entries, 0 to 45417
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      45418 non-null  int64         
 1   created_date  45418 non-null  datetime64[ns]
 2   total_paid    45418 non-null  float64       
 3   state         45418 non-null  object        
 4   year          45418 non-null  int32         
 5   month         45418 non-null  period[M]     
dtypes: datetime64[ns](1), float64(1), int32(1), int64(1), object(1), period[M](1)
memory usage: 1.9+ MB


In [30]:
orderlines_analyse.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60179 entries, 0 to 60178
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                60179 non-null  int64  
 1   id_order          60179 non-null  int64  
 2   product_quantity  60179 non-null  int64  
 3   sku               60179 non-null  object 
 4   unit_price        60179 non-null  float64
 5   date              60179 non-null  object 
dtypes: float64(1), int64(3), object(2)
memory usage: 2.8+ MB


In [31]:
orders_orderlines_products_analyse['umsatz'] = (
    orders_orderlines_products_analyse['product_quantity'] * orders_orderlines_products_analyse['unit_price']
)

###5.4 Rabattanalyse – Kernfrage
**Nimmt der Umsatz mit steigenden Rabatten zu?**

In [32]:
m = orders_orderlines_products_analyse
m['rabatt_bucket'] = pd.cut(m['discount_percent'], bins=range(0, 110, 10))

In [33]:
m.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60179 entries, 0 to 60178
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   order_id          60179 non-null  int64   
 1   created_date      60179 non-null  object  
 2   total_paid        60179 non-null  float64 
 3   state             60179 non-null  object  
 4   id                60179 non-null  int64   
 5   id_order          60179 non-null  int64   
 6   product_quantity  60179 non-null  int64   
 7   sku               60179 non-null  object  
 8   unit_price        60179 non-null  float64 
 9   date              60179 non-null  object  
 10  name              60179 non-null  object  
 11  desc              60179 non-null  object  
 12  price             60179 non-null  float64 
 13  promo_price       60179 non-null  object  
 14  in_stock          60179 non-null  int64   
 15  type              60179 non-null  object  
 16  discount_percent  6017

In [34]:
m['rabatt_bucket'].unique()

[(0.0, 10.0], (40.0, 50.0], (20.0, 30.0], (10.0, 20.0], NaN, ..., (90, 100], (80, 90], (60, 70], (50, 60], (70, 80]]
Length: 11
Categories (10, interval[int64, right]): [(0, 10] < (10, 20] < (20, 30] < (30, 40] ... (60, 70] <
                                          (70, 80] < (80, 90] < (90, 100]]

In [35]:
sales_df = m.copy()
sales_df = sales_df.drop(columns=['promo_price','id', 'in_stock'])


In [36]:
sales_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60179 entries, 0 to 60178
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   order_id          60179 non-null  int64   
 1   created_date      60179 non-null  object  
 2   total_paid        60179 non-null  float64 
 3   state             60179 non-null  object  
 4   id_order          60179 non-null  int64   
 5   product_quantity  60179 non-null  int64   
 6   sku               60179 non-null  object  
 7   unit_price        60179 non-null  float64 
 8   date              60179 non-null  object  
 9   name              60179 non-null  object  
 10  desc              60179 non-null  object  
 11  price             60179 non-null  float64 
 12  type              60179 non-null  object  
 13  discount_percent  60179 non-null  float64 
 14  category          60179 non-null  object  
 15  umsatz            60179 non-null  float64 
 16  rabatt_bucket     5601

In [38]:
#Bereinigte Daten speichern in Drive

from google.colab import files

sales_df.to_csv("sales_qu.csv", index=False)
files.download("sales_qu.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>